# arXiv computer-science classification with frozen BERT

**Task:** predict a paper’s primary category from its title and abstract: `cs.CL` (Computation and Language), `cs.CV` (Computer Vision), or `cs.CR` (Cryptography and Security).

Raw text → BERT tokenizer → **frozen bert-base-uncased** → final-layer `[CLS]` embedding → separate logistic-regression classification head → label.

No starter notebook or response prompts were supplied; this notebook provides completed steps and short interpretation prompts. Data is real arXiv metadata fetched from the public API (100 most recent matches per category), retaining only papers with that primary category. Cross-listings are not treated as extra labels. This is a small, recent convenience sample, not a representative sample of all CS research.

Sources: [arXiv API](https://info.arxiv.org/help/api/user-manual.html), [BERT checkpoint](https://huggingface.co/google-bert/bert-base-uncased), [BERT outputs](https://huggingface.co/docs/transformers/model_doc/bert).

## Environment
Install `requirements.txt` in a Python environment with a Jupyter kernel, then run all cells. The first run downloads the pretrained checkpoint. The dataset is included; no Kaggle credentials are needed. All results below are produced by execution.

In [1]:
from pathlib import Path
import hashlib, json, random
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(4)
DATA = Path('data/arxiv_cs.json')
if not DATA.exists(): DATA = Path('notebooks/data/arxiv_cs.json')
df = pd.DataFrame(json.loads(DATA.read_text()))
df['text'] = df.title + '\n\n' + df.abstract
assert not df.text.duplicated().any()
print('Dataset SHA256:', hashlib.sha256(DATA.read_bytes()).hexdigest())
print('Papers:', len(df))
print(df.label.value_counts().sort_index().to_string())

/private/tmp/arxiv-bert-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset SHA256: e3cd79d3ac2664259c1eec82580d4e20fe4bcf8ee0afede0f09d400a1ac07632
Papers: 217
label
cs.CL    71
cs.CR    66
cs.CV    80


## Split before training
Use a stratified 60/20/20 train/validation/test split. Validation selects regularization strength; test labels are used only for the final evaluation. The scaler is fitted only on training embeddings.

In [2]:
indices = np.arange(len(df))
train_idx, remaining = train_test_split(indices, test_size=0.4, stratify=df.label, random_state=SEED)
val_idx, test_idx = train_test_split(remaining, test_size=0.5, stratify=df.label.iloc[remaining], random_state=SEED)
assert not (set(train_idx) & set(val_idx) or set(train_idx) & set(test_idx) or set(val_idx) & set(test_idx))
print('Train / validation / test:', len(train_idx), len(val_idx), len(test_idx))

Train / validation / test: 130 43 44


In [3]:
MODEL = 'google-bert/bert-base-uncased'
# Pin the pretrained checkpoint for reproducibility.
REVISION = '86b5e0934494bd15c9632b12f734a8a67f723594'
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
bert = AutoModel.from_pretrained(MODEL, revision=REVISION)
bert.requires_grad_(False)
bert.eval()  # Disable dropout as well as freezing all parameters.
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
bert.to(device)
def fingerprint(model):
    digest = hashlib.sha256()
    for name, tensor in model.state_dict().items():
        digest.update(name.encode())
        digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()
before = fingerprint(bert)
assert all(not p.requires_grad for p in bert.parameters())
print('Device:', device, '| Trainable BERT parameters:', sum(p.numel() for p in bert.parameters() if p.requires_grad))

Device: mps | Trainable BERT parameters: 0


In [4]:
MAX_LENGTH, BATCH_SIZE = 256, 8
lengths = [len(tokenizer.encode(text, truncation=False)) for text in df.text]
print(f'Truncated texts: {sum(length > MAX_LENGTH for length in lengths)}/{len(lengths)}')
features = []
with torch.inference_mode():
    for start in range(0, len(df), BATCH_SIZE):
        tokens = tokenizer(df.text.iloc[start:start+BATCH_SIZE].tolist(), padding=True,
                           truncation=True, max_length=MAX_LENGTH, return_tensors='pt').to(device)
        outputs = bert(**tokens)
        # Raw final hidden state at CLS; do not use pooler_output or mean pooling.
        features.append(outputs.last_hidden_state[:, 0, :].cpu().numpy())
X = np.concatenate(features)
y = df.label.to_numpy()
assert X.shape == (len(df), 768)
assert np.isfinite(X).all()
print('Frozen CLS embedding matrix:', X.shape)

Truncated texts: 176/217


Frozen CLS embedding matrix: (217, 768)


**Interpretation — what is trained?** Each text becomes a contextual 768-dimensional vector. BERT is used only for inference: all parameters have `requires_grad=False`, dropout is disabled, and embedding extraction has no autograd graph. The independently trained multinomial logistic-regression head learns category boundaries. Its scaler and coefficients can change; BERT cannot.

In [5]:
best, best_score, best_c = None, -1, None
for c in [0.01, 0.1, 1.0, 10.0]:
    candidate = make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=3000, random_state=SEED))
    candidate.fit(X[train_idx], y[train_idx])
    score = f1_score(y[val_idx], candidate.predict(X[val_idx]), average='macro')
    print(f'C={c:g}: validation macro-F1={score:.4f}')
    if score > best_score:
        best, best_score, best_c = candidate, score, c
print('Selected C:', best_c)
# Keep the selected head fitted on training only for a clear held-out comparison.
pred = best.predict(X[test_idx])
baseline = DummyClassifier(strategy='most_frequent').fit(X[train_idx], y[train_idx])
base_pred = baseline.predict(X[test_idx])
metrics = {'test_accuracy': accuracy_score(y[test_idx], pred),
           'test_macro_f1': f1_score(y[test_idx], pred, average='macro'),
           'baseline_accuracy': accuracy_score(y[test_idx], base_pred),
           'baseline_macro_f1': f1_score(y[test_idx], base_pred, average='macro')}
print(json.dumps(metrics, indent=2))
print(classification_report(y[test_idx], pred, digits=3, zero_division=0))
labels = sorted(df.label.unique())
display(pd.DataFrame(confusion_matrix(y[test_idx], pred, labels=labels),
                     index=['actual '+s for s in labels], columns=['predicted '+s for s in labels]))
after = fingerprint(bert)
assert before == after, 'BERT parameters or buffers changed!'
assert all(p.grad is None and not p.requires_grad for p in bert.parameters())
print('BERT SHA256 before:', before)
print('BERT SHA256 after: ', after)
print('BERT unchanged:', before == after)

C=0.01: validation macro-F1=0.7823
C=0.1: validation macro-F1=0.8052
C=1: validation macro-F1=0.8052
C=10: validation macro-F1=0.7745
Selected C: 0.1
{
  "test_accuracy": 0.6818181818181818,
  "test_macro_f1": 0.6860494344365312,
  "baseline_accuracy": 0.36363636363636365,
  "baseline_macro_f1": 0.17777777777777778
}
              precision    recall  f1-score   support

       cs.CL      0.625     0.667     0.645        15
       cs.CR      0.889     0.615     0.727        13
       cs.CV      0.632     0.750     0.686        16

    accuracy                          0.682        44
   macro avg      0.715     0.677     0.686        44
weighted avg      0.705     0.682     0.684        44



,predicted cs.CL,predicted cs.CR,predicted cs.CV
actual cs.CL,10,1,4
actual cs.CR,2,8,3
actual cs.CV,4,0,12


BERT SHA256 before: 862f2ad8eb15a3aa7a579875c38d422b142447652259f4d1f93c00a0c2114805
BERT SHA256 after:  862f2ad8eb15a3aa7a579875c38d422b142447652259f4d1f93c00a0c2114805
BERT unchanged: True


In [6]:
errors = df.iloc[test_idx][['title', 'label']].copy()
errors['prediction'] = pred
display(errors[errors.label != errors.prediction].head(8))
print(f"Interpretation: test accuracy is {metrics['test_accuracy']:.1%} and macro-F1 is {metrics['test_macro_f1']:.3f}, "
      f"compared with majority baseline accuracy {metrics['baseline_accuracy']:.1%} "
      f"and macro-F1 {metrics['baseline_macro_f1']:.3f}. "
      'Macro-F1 weights each category equally; accuracy weights each paper equally.')

,title,label,prediction
76,Back to Basics: A Skill-Centric Approach to Ge...,cs.CV,cs.CL
175,Maximum-Likelihood Decoding of Gabidulin Codes...,cs.CR,cs.CV
199,Holo-NN: Holographic Nearest Neighbor Search v...,cs.CR,cs.CV
147,Seeing Before Naming: Context-Grounded Learnin...,cs.CV,cs.CL
24,The Geometry of Emotion Representations Across...,cs.CL,cs.CV
165,The Target Agent Is Its Own Best Critic but a ...,cs.CR,cs.CL
158,PRISMA: Differentially Private Synthetic In-Ca...,cs.CR,cs.CV
21,Probing the phon* representations of the Mimi ...,cs.CL,cs.CV


Interpretation: test accuracy is 68.2% and macro-F1 is 0.686, compared with majority baseline accuracy 36.4% and macro-F1 0.178. Macro-F1 weights each category equally; accuracy weights each paper equally.


**Interpretation — limitations and improvements.** The confusion matrix shows which categories the head confuses, and the error table provides concrete examples to inspect. Papers may legitimately span multiple categories even though this task assigns only their primary label. The small recent sample and random split limit generalization: related papers can share vocabulary across splits, and long abstracts lose text after 256 tokens. A larger sample, temporal holdout, duplicate-family checks, and a TF–IDF baseline would strengthen the comparison. Head-only hyperparameter tuning and longer input lengths are permitted improvements; updating any BERT parameter would violate this assignment.